# NV5 — Dự báo occupancy cổng và session arrivals

Toàn bộ mã nguồn NV5 nằm trong notebook này; thư mục scripts dành cho chương 4. Các cell dưới đây nạp dữ liệu NV1, tạo chuỗi theo cụm trạm, fit baseline và xuất artefact. Đọc reports/NV5_demand_forecasting.md để xem định nghĩa mục 3.5.1–3.5.2.

Trước khi chạy cell cuối, nhóm cần điền mapping port_id,cluster_id đã thống nhất vào data/processed/station_cluster_map.csv, đồng thời tạo reports/nv5_splits.json theo mốc train/validation/test do Khải cung cấp. Notebook không tự quyết định cluster hay cách chia tập.

## 1. Thiết lập và thư viện

In [25]:
"""NV5: 15-minute station-cluster port occupancy and session-arrival forecasts.

Run each notebook cell in order after the team supplies the approved station-cluster
mapping and split milestones.

The mapping file must contain port_id,cluster_id. The split JSON must contain
train_end_utc, validation_end_utc, and test_end_utc (exclusive boundaries).
"""
from __future__ import annotations

import argparse
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

ROOT = Path.cwd()
while not (ROOT / 'src' / 'config.py').exists():
    if ROOT.parent == ROOT:
        raise RuntimeError('Open this notebook from the project workspace')
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.config import (RAW_DATA_FILE, SESSIONS_CLEAN_PARQUET,
                        SESSIONS_FEATURES_PARQUET)

OUT = ROOT / "outputs" / "nv5"
STEP = pd.Timedelta(minutes=15)
SEED = 42


def _as_datetime(series):
    return pd.to_datetime(series, errors="coerce", utc=True).dt.tz_convert(None)


def _expected_port_id(df):
    mac = df["MAC Address"].astype("string").str.strip()
    port = df["Port Number"].astype("string").str.strip()
    good = mac.notna() & port.notna() & mac.ne("") & port.ne("")
    result = pd.Series(pd.NA, index=df.index, dtype="string")
    result.loc[good] = mac.loc[good] + "_Port_" + port.loc[good]
    return result


## 2. Nạp dữ liệu NV1, kiểm tra port_id và gán cluster

In [26]:
def load_sessions():
    """Prefer NV1 features, compare its row/valid counts with clean parquet."""
    parquet_rejection = None
    if SESSIONS_FEATURES_PARQUET.exists():
        try:
            features = pd.read_parquet(SESSIONS_FEATURES_PARQUET)
            clean = (pd.read_parquet(SESSIONS_CLEAN_PARQUET)
                     if SESSIONS_CLEAN_PARQUET.exists() else None)
            if clean is not None:
                k = ["MAC Address", "Port Number", "Start Date", "End Date"]
                audit = {
                    "clean_rows": len(clean), "features_rows": len(features),
                    "clean_valid_sessions": int(clean["is_valid_session"].sum()),
                    "features_valid_sessions": int(features["is_valid_session"].sum()),
                    "same_session_keys": bool(clean[k].sort_values(k).reset_index(drop=True).equals(
                        features[k].sort_values(k).reset_index(drop=True))),
                }
                if (audit["clean_rows"] != audit["features_rows"] or
                    audit["clean_valid_sessions"] != audit["features_valid_sessions"] or
                    not audit["same_session_keys"]):
                    raise ValueError(f"NV1 clean/features mismatch: {audit}")
            else:
                audit = {"clean_parquet_available": False}
            print("Source: NV1 sessions_features.parquet; audit:", audit)
            return features, "nv1_parquet", audit
        except (ImportError, ValueError, OSError) as exc:
            parquet_rejection = str(exc)
            print("NV1 parquet unavailable/inconsistent:", str(exc)[:300])

    from src.feature_engineering import build_clean_pipeline
    print("Fallback: rebuild NV1 cleaning flags from raw CSV")
    raw = pd.read_csv(RAW_DATA_FILE, low_memory=False)
    data, initial, duplicates = build_clean_pipeline(raw)
    audit = {"raw_rows": initial, "exact_duplicates_removed": duplicates,
             "clean_rows": len(data),
             "valid_sessions": int(data["is_valid_session"].sum()),
             "parquet_rejection_reason": parquet_rejection}
    return data, "nv1_raw_recomputed", audit


def attach_clusters(df, mapping_path):
    """Apply explicit port-to-station-cluster mapping; never infer cluster IDs."""
    if "MAC Address" not in df or "Port Number" not in df:
        raise ValueError("NV1 source must include MAC Address and Port Number")
    expected = _expected_port_id(df)
    if "port_id" in df:
        actual = df["port_id"].astype("string")
        mismatch = expected.notna() & actual.notna() & expected.ne(actual)
        if mismatch.any():
            raise ValueError(f"port_id disagrees with MAC Address + Port Number on {int(mismatch.sum())} rows")
        missing_components_with_id = expected.isna() & actual.notna()
        if missing_components_with_id.any():
            raise ValueError("port_id is populated where MAC Address/Port Number is missing")
    df = df.copy()
    df["port_id"] = expected
    pairs = df.loc[df.port_id.notna(), ["port_id", "MAC Address", "Port Number"]].drop_duplicates()
    if pairs.port_id.duplicated().any():
        raise ValueError("A port_id maps to multiple MAC Address + Port Number pairs")
    pair_to_id = pairs.groupby(["MAC Address", "Port Number"], dropna=False).port_id.nunique()
    if pair_to_id.gt(1).any():
        raise ValueError("A physical MAC Address + Port Number pair maps to multiple port_id values")

    mapping_path = Path(mapping_path)
    if not mapping_path.exists():
        template = (df.loc[df.port_id.notna(), "port_id"].astype(str)
                    .drop_duplicates().sort_values().to_frame())
        template["cluster_id"] = ""
        mapping_path.parent.mkdir(parents=True, exist_ok=True)
        template.to_csv(mapping_path, index=False)
        raise FileNotFoundError(
            f"Created blank mapping template at {mapping_path}; have the group assign cluster_id values"
        )
    mapping = pd.read_csv(mapping_path, dtype="string")
    required = {"port_id", "cluster_id"}
    if not required.issubset(mapping.columns):
        raise ValueError("Cluster mapping requires columns port_id,cluster_id")
    mapping = mapping[["port_id", "cluster_id"]].dropna()
    mapping["port_id"] = mapping.port_id.str.strip()
    mapping["cluster_id"] = mapping.cluster_id.str.strip()
    if mapping[["port_id", "cluster_id"]].eq("").any().any():
        raise ValueError("Cluster mapping contains blank IDs")
    if mapping.port_id.duplicated().any():
        raise ValueError("Each physical port_id must map to exactly one cluster_id")
    known_ports = set(df.port_id.dropna().astype(str))
    unknown = set(mapping.port_id) - known_ports
    if unknown:
        raise ValueError(f"Cluster mapping contains {len(unknown)} port IDs absent from NV1 data")
    unmapped = known_ports - set(mapping.port_id)
    if unmapped:
        raise ValueError(f"Cluster mapping is missing {len(unmapped)} observed physical ports")
    df["cluster_id"] = df.port_id.map(mapping.set_index("port_id").cluster_id)
    df.attrs["cluster_mapping"] = {"mapped_ports": int(mapping.port_id.nunique()),
                                     "unmapped_ports": sorted(known_ports-set(mapping.port_id))}
    return df


def make_regular_series(df):
    """Create cluster-local counts on a regular UTC grid using [start,end)."""
    required = ["start_datetime_utc", "end_datetime_utc", "is_valid_session",
                "port_id", "cluster_id"]
    absent = [c for c in required if c not in df]
    if absent:
        raise ValueError(f"Missing required columns: {absent}")
    valid_source = df.loc[df.is_valid_session.fillna(False)].copy()
    missing_port_count = int(valid_source.port_id.isna().sum())
    missing_cluster_count = int(valid_source.cluster_id.isna().sum())
    data = valid_source.copy()
    data["start_datetime_utc"] = _as_datetime(data.start_datetime_utc)
    data["end_datetime_utc"] = _as_datetime(data.end_datetime_utc)
    nonpositive_intervals = int((data.end_datetime_utc <= data.start_datetime_utc).fillna(False).sum())
    data = data.dropna(subset=["start_datetime_utc", "end_datetime_utc", "cluster_id"])
    data = data.loc[data.end_datetime_utc > data.start_datetime_utc]
    if data.empty:
        raise ValueError("No valid, mapped sessions remain")
    left = data.start_datetime_utc.min().floor("15min")
    right = data.end_datetime_utc.max().ceil("15min")
    index = pd.date_range(left, right - STEP, freq="15min")
    frames = []
    for cluster, group in data.groupby("cluster_id", sort=True):
        # Invalid/missing port IDs do not count toward physical occupancy.
        known = group.loc[group.port_id.notna() & group.port_id.astype("string").str.strip().ne("")]
        occupancy = np.zeros(len(index), dtype=np.int64)
        # Each port contributes at most 1 in a bin, even when sessions overlap.
        for _, port_sessions in known.groupby("port_id", sort=False):
            diffs = np.zeros(len(index) + 1, dtype=np.int64)
            starts = ((port_sessions.start_datetime_utc - left) // STEP).to_numpy(dtype=int)
            ends = ((port_sessions.end_datetime_utc - left - pd.Timedelta(1, "ns")) // STEP + 1).to_numpy(dtype=int)
            starts = np.clip(starts, 0, len(index))
            ends = np.clip(ends, 0, len(index))
            np.add.at(diffs, starts, 1)
            np.add.at(diffs, ends, -1)
            occupancy += np.cumsum(diffs[:-1]) > 0

        arrivals = np.zeros(len(index), dtype=np.int64)
        starts = ((group.start_datetime_utc - left) // STEP).to_numpy(dtype=int)
        in_range = (starts >= 0) & (starts < len(index))
        np.add.at(arrivals, starts[in_range], 1)
        frames.append(pd.DataFrame({"cluster_id": cluster, "timestamp_utc": index,
                                    "occupancy_ports": occupancy,
                                    "session_arrivals": arrivals,
                                    "observed_physical_ports": int(known.port_id.nunique())}))
    result = pd.concat(frames, ignore_index=True)
    info = {"valid_mapped_sessions": int(len(data)),
            "valid_sessions_excluded_nonpositive_interval": nonpositive_intervals,
            "valid_sessions_missing_port_id": missing_port_count,
            "valid_sessions_without_cluster_id": missing_cluster_count,
            "observed_physical_ports": int(data.port_id.nunique()),
            "first_utc": str(left), "last_utc_exclusive": str(right),
            "time_bins_per_cluster": int(len(index)),
            "clusters": sorted(data.cluster_id.astype(str).unique().tolist()),
            "max_occupancy": int(result.occupancy_ports.max()),
            "occupancy_exceeds_observed_ports": bool((result.occupancy_ports > result.observed_physical_ports).any())}
    if info["occupancy_exceeds_observed_ports"]:
        raise AssertionError("Cluster occupancy exceeds its observed physical port count")
    return result, info


## 3. Tạo chuỗi occupancy và arrivals theo cụm

In [27]:
def add_features(series, target):
    frames = []
    for cluster, group in series.groupby("cluster_id", sort=False):
        g = group.sort_values("timestamp_utc").set_index("timestamp_utc")
        s = g[[target]].copy()
        t = s.index
        slot = t.hour * 4 + t.minute // 15
        dow = t.dayofweek
        s["slot"] = slot.astype(int)
        s["dow"] = dow.astype(int)
        s["month"] = t.month.astype(int)
        s["weekend"] = (dow >= 5).astype(int)
        s["hour_sin"] = np.sin(2*np.pi*slot/96)
        s["hour_cos"] = np.cos(2*np.pi*slot/96)
        s["week_sin"] = np.sin(2*np.pi*dow/7)
        s["week_cos"] = np.cos(2*np.pi*dow/7)
        for lag in (1, 4, 96, 672):
            s[f"lag_{lag}"] = s[target].shift(lag)
        for window in (4, 96, 672):
            s[f"rolling_mean_{window}"] = s[target].shift(1).rolling(window, min_periods=window).mean()
        s = s.dropna().copy()
        s["cluster_id"] = cluster
        frames.append(s.reset_index(names="timestamp_utc"))
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


## 4. Đặc trưng trễ và hàm đánh giá

In [28]:
def metrics(y, p):
    return {"MAE": float(mean_absolute_error(y, p)),
            "RMSE": float(np.sqrt(mean_squared_error(y, p))),
            "R2": float(r2_score(y, p))}


def _parse_splits(splits):
    train_end, validation_end, test_end = [
        pd.to_datetime(splits[key], utc=True).tz_convert(None)
        for key in ("train_end_utc", "validation_end_utc", "test_end_utc")
    ]
    if not train_end < validation_end < test_end:
        raise ValueError("Require train_end_utc < validation_end_utc < test_end_utc")
    return train_end, validation_end, test_end


def ridge_predictions(train, validation, test, features, target):
    """Fit the scaler and Ridge only on train; transform later splits unchanged."""
    scaler = StandardScaler().fit(train[features])
    model = Ridge(alpha=10.0).fit(scaler.transform(train[features]), train[target])
    return (model.predict(scaler.transform(validation[features])),
            model.predict(scaler.transform(test[features])), scaler)


def run_target(series, target, splits):
    """Run per-cluster baselines/models with externally supplied split boundaries."""
    from xgboost import XGBRegressor
    train_end, validation_end, test_end = _parse_splits(splits)
    data = add_features(series, target)
    cols = [c for c in data.columns if c not in {target, "cluster_id", "timestamp_utc"}]
    records, prediction_rows = [], []
    for cluster, d in data.groupby("cluster_id", sort=True):
        d = d.sort_values("timestamp_utc")
        train = d.loc[d.timestamp_utc < train_end]
        val = d.loc[(d.timestamp_utc >= train_end) & (d.timestamp_utc < validation_end)]
        test = d.loc[(d.timestamp_utc >= validation_end) & (d.timestamp_utc < test_end)]
        if train.empty or val.empty or test.empty:
            raise ValueError(f"Cluster {cluster!r} has an empty train/validation/test split")
        hist = train.groupby(["dow", "slot"])[target].mean()
        default = float(train[target].mean())
        prediction_sets = {
            "Historical Mean": {part: np.array([hist.get((int(row.dow), int(row.slot)), default)
                                             for row in part_d.itertuples()])
                                for part, part_d in (("validation", val), ("test", test))},
            "Persistence Day t-96": {"validation": val.lag_96.to_numpy(), "test": test.lag_96.to_numpy()},
            "Persistence Week t-672": {"validation": val.lag_672.to_numpy(), "test": test.lag_672.to_numpy()},
        }
        models = {
            "Ridge": Ridge(alpha=10.0),
            "Random Forest": RandomForestRegressor(n_estimators=80, max_depth=14,
                min_samples_leaf=4, n_jobs=-1, random_state=SEED),
            "XGBoost": XGBRegressor(n_estimators=220, max_depth=6, learning_rate=.06,
                subsample=.85, colsample_bytree=.9, tree_method="hist",
                objective="reg:squarederror", n_jobs=4, random_state=SEED),
        }
        for name, model in models.items():
            if name == "Ridge":
                pred_val, pred_test, _ = ridge_predictions(train, val, test, cols, target)
            else:
                model.fit(train[cols], train[target])
                pred_val = model.predict(val[cols])
                pred_test = model.predict(test[cols])
            prediction_sets[name] = {"validation": np.maximum(0, pred_val),
                                     "test": np.maximum(0, pred_test)}
        for name, parts in prediction_sets.items():
            for split_name, part in (("validation", val), ("test", test)):
                pred = np.maximum(0, np.asarray(parts[split_name], dtype=float))
                y = part[target].to_numpy(dtype=float)
                records.append({"cluster_id": cluster, "target": target, "model": name,
                                "split": split_name, "train_rows": len(train),
                                "validation_rows": len(val), "test_rows": len(test),
                                "train_end_utc": str(train_end), "validation_end_utc": str(validation_end),
                                "test_end_utc": str(test_end), **metrics(y, pred)})
                prediction_rows.extend({"cluster_id": cluster,
                    "timestamp_utc": ts.isoformat(), "target": target, "model": name,
                    "split": split_name, "y_true": float(actual), "y_pred": float(estimate)}
                    for ts, actual, estimate in zip(part.timestamp_utc, y, pred))
    score = pd.DataFrame(records)
    preds = pd.DataFrame(prediction_rows)
    return score, preds


## 5. Baseline, Ridge chuẩn hóa train-only, Random Forest và XGBoost

In [29]:
def main(argv=None):
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--cluster-map", required=True,
                        help="Group-approved CSV with port_id,cluster_id")
    parser.add_argument("--splits-json", required=True,
                        help="JSON with train_end_utc, validation_end_utc, test_end_utc")
    args = parser.parse_args(argv)
    OUT.mkdir(parents=True, exist_ok=True)
    df, provenance, source_audit = load_sessions()
    df = attach_clusters(df, args.cluster_map)
    series, series_audit = make_regular_series(df)
    series_audit.update({"source": provenance, **source_audit,
                         "cluster_mapping": df.attrs.get("cluster_mapping", {})})
    pd.Series(series_audit).to_json(OUT / "data_audit.json", indent=2, force_ascii=False)
    series[["cluster_id", "timestamp_utc", "occupancy_ports", "session_arrivals"]].to_csv(
        OUT / "timeseries_15min.csv", index=False)
    with open(args.splits_json, encoding="utf-8") as f:
        splits = json.load(f)
    score_parts, prediction_parts = [], []
    for target in ("occupancy_ports", "session_arrivals"):
        scores, predictions = run_target(series, target, splits)
        score_parts.append(scores)
        prediction_parts.append(predictions)
    comparison = pd.concat(score_parts, ignore_index=True)
    predictions = pd.concat(prediction_parts, ignore_index=True)
    comparison.to_csv(OUT / "nv5_model_comparison.csv", index=False)
    predictions.to_csv(OUT / "baseline_predictions.csv", index=False)
    print("NV5 metrics:\n", comparison.to_string(index=False))
    print("Artifacts:", OUT)


## 6. Kiểm tra dữ liệu đầu vào (chạy được ngay)

Cell này hiện kết quả từ NV1 và lưu audit mà không cần mapping cụm hay mốc chia. Đây là kiểm tra dữ liệu, chưa phải metric dự báo theo cụm.


In [30]:
sessions, data_source, source_audit = load_sessions()
valid_sessions = sessions.loc[sessions.is_valid_session.fillna(False)].copy()
valid_start = _as_datetime(valid_sessions["start_datetime_utc"])
valid_end = _as_datetime(valid_sessions["end_datetime_utc"])
expected_ids = _expected_port_id(sessions)
stored_ids = sessions["port_id"].astype("string")
id_comparable = expected_ids.notna() & stored_ids.notna()
id_mismatches = int((expected_ids.loc[id_comparable] != stored_ids.loc[id_comparable]).sum())
port_pairs = sessions.loc[sessions.port_id.notna(), ["port_id", "MAC Address", "Port Number"]].drop_duplicates()
port_pair_conflicts = int(port_pairs.groupby("port_id")[["MAC Address", "Port Number"]].nunique().gt(1).any(axis=1).sum())
cluster_map_path = ROOT / "data" / "processed" / "station_cluster_map.csv"
splits_path = ROOT / "reports" / "nv5_splits.json"
cluster_map_ready = False
if cluster_map_path.exists():
    map_preview = pd.read_csv(cluster_map_path, dtype="string")
    cluster_map_ready = {"port_id", "cluster_id"}.issubset(map_preview.columns) and not map_preview[["port_id", "cluster_id"]].isna().any().any() and not map_preview[["port_id", "cluster_id"]].apply(lambda c: c.str.strip().eq("")).any().any()
audit = {
    "data_source": data_source,
    **source_audit,
    "valid_sessions": int(len(valid_sessions)),
    "valid_zero_duration_sessions": int((valid_end == valid_start).sum()),
    "valid_negative_duration_sessions": int((valid_end < valid_start).sum()),
    "valid_sessions_missing_port_id": int(valid_sessions.port_id.isna().sum()),
    "observed_port_ids": int(sessions.port_id.nunique()),
    "port_id_mismatches_vs_mac_port": id_mismatches,
    "port_ids_mapping_to_multiple_physical_pairs": port_pair_conflicts,
    "cluster_map_ready": bool(cluster_map_ready),
    "split_milestones_file_present": bool(splits_path.exists()),
}
OUT.mkdir(parents=True, exist_ok=True)
pd.Series(audit).to_json(OUT / "nv5_input_audit.json", indent=2, force_ascii=False)
from IPython.display import display
display(pd.DataFrame({"check": list(audit), "result": list(audit.values())}))
print("Input audit saved to:", OUT / "nv5_input_audit.json")


Source: NV1 sessions_features.parquet; audit: {'clean_rows': 259411, 'features_rows': 259411, 'clean_valid_sessions': 259356, 'features_valid_sessions': 259356, 'same_session_keys': True}


,check,result
0,data_source,nv1_parquet
1,clean_rows,259411
2,features_rows,259411
3,clean_valid_sessions,259356
4,features_valid_sessions,259356
5,same_session_keys,True
6,valid_sessions,259356
7,valid_zero_duration_sessions,12
8,valid_negative_duration_sessions,0
9,valid_sessions_missing_port_id,0


Input audit saved to: e:\backup\class\s5\business analyst\doan\project\ev-charging-analytics---IS403\outputs\nv5\nv5_input_audit.json


## 7. Chạy thí nghiệm và lưu kết quả

Chạy cell sau cùng sau khi hai đầu vào của nhóm đã sẵn sàng. Kết quả được ghi vào outputs/nv5/.


In [31]:
cluster_map = ROOT / 'data' / 'processed' / 'station_cluster_map.csv'
splits_json = ROOT / 'reports' / 'nv5_splits.json'
missing_inputs = []
if not cluster_map.exists():
    missing_inputs.append(f'Chưa có {cluster_map}. Sao chép reports/station_cluster_map_template.csv tới vị trí này và điền cluster_id theo thống nhất của nhóm.')
if not splits_json.exists():
    missing_inputs.append(f'Chưa có {splits_json}. Tạo JSON với train_end_utc, validation_end_utc, test_end_utc theo mốc Khải cung cấp.')
if cluster_map.exists():
    mapping_check = pd.read_csv(cluster_map, dtype='string')
    required_map_columns = {'port_id', 'cluster_id'}
    if not required_map_columns.issubset(mapping_check.columns):
        missing_inputs.append('Cluster map phải có hai cột port_id,cluster_id.')
    elif mapping_check[list(required_map_columns)].isna().any().any() or mapping_check[list(required_map_columns)].apply(lambda col: col.str.strip().eq('')).any().any():
        missing_inputs.append('Cluster map còn cluster_id/port_id trống; hãy điền đầy đủ theo quyết định của nhóm.')
if splits_json.exists():
    try:
        with open(splits_json, encoding='utf-8') as split_file:
            _parse_splits(json.load(split_file))
    except (KeyError, ValueError, TypeError) as error:
        missing_inputs.append(f'Mốc chia trong {splits_json} chưa hợp lệ: {error}')
if missing_inputs:
    print('Chưa chạy NV5 vì còn thiếu đầu vào bắt buộc:')
    print('\n'.join(f'- {message}' for message in missing_inputs))
else:
    main(['--cluster-map', str(cluster_map), '--splits-json', str(splits_json)])


Chưa chạy NV5 vì còn thiếu đầu vào bắt buộc:
- Chưa có e:\backup\class\s5\business analyst\doan\project\ev-charging-analytics---IS403\reports\nv5_splits.json. Tạo JSON với train_end_utc, validation_end_utc, test_end_utc theo mốc Khải cung cấp.
- Cluster map còn cluster_id/port_id trống; hãy điền đầy đủ theo quyết định của nhóm.
